<a href="https://colab.research.google.com/github/Yucheol-Son-BYUI/CSE310_W0_HelloWorld/blob/main/Module5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# FULL COLAB SETUP (Google Drive -> unzip -> merge -> tf.data)
# Put training1.zip and training2.zip in MyDrive (or adjust paths)
# ============================================================

import os
import zipfile
import shutil
import tensorflow as tf
from google.colab import drive

# ----------------------------
# 0) CONFIG
# ----------------------------
ZIP_PATHS = [
    "/content/training1.zip",
    "/content/training2.zip",
]

EXTRACT_ROOT = "/content"        # fast local runtime disk
TRAIN1_DIR = os.path.join(EXTRACT_ROOT, "training1")
TRAIN2_DIR = os.path.join(EXTRACT_ROOT, "training2")
MERGED_DIR = os.path.join(EXTRACT_ROOT, "all")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
VAL_SPLIT = 0.2
SEED = 123

# ----------------------------
# 2) VERIFY ZIP FILES EXIST
# ----------------------------
for zp in ZIP_PATHS:
    print("ZIP:", zp, "exists:", os.path.exists(zp))
    if not os.path.exists(zp):
        raise FileNotFoundError(f"Missing zip: {zp}\nUpload it to Drive or fix ZIP_PATHS.")

# ----------------------------
# 3) UNZIP (ONLY IF NEEDED)
# ----------------------------
os.makedirs(EXTRACT_ROOT, exist_ok=True)

need_unzip = not (os.path.exists(TRAIN1_DIR) and os.path.exists(TRAIN2_DIR))
print("Need unzip:", need_unzip)

if need_unzip:
    # optional: clean old extract root
    # shutil.rmtree(EXTRACT_ROOT)
    # os.makedirs(EXTRACT_ROOT, exist_ok=True)

    for zp in ZIP_PATHS:
        print("Extracting:", zp)
        with zipfile.ZipFile(zp, "r") as z:
            z.extractall(EXTRACT_ROOT)

print("Extract root contents:", os.listdir(EXTRACT_ROOT))

# ----------------------------
# 4) MERGE training1 + training2 -> all (ONLY IF NEEDED)
# ----------------------------
os.makedirs(MERGED_DIR, exist_ok=True)

def is_merged_ok():
    # merged ok if it has class folders with at least some images
    if not os.path.exists(MERGED_DIR):
        return False
    classes = [d for d in os.listdir(MERGED_DIR) if os.path.isdir(os.path.join(MERGED_DIR, d))]
    if len(classes) == 0:
        return False
    # check at least one class has images
    for c in classes[:5]:
        if len(os.listdir(os.path.join(MERGED_DIR, c))) > 0:
            return True
    return False

if not is_merged_ok():
    print("Merging into:", MERGED_DIR)

    srcs = [TRAIN1_DIR, TRAIN2_DIR]
    for src in srcs:
        if not os.path.exists(src):
            raise FileNotFoundError(f"Missing extracted folder: {src}\nCheck zip contents/paths.")

        for class_name in os.listdir(src):
            src_class = os.path.join(src, class_name)
            if not os.path.isdir(src_class):
                continue

            dst_class = os.path.join(MERGED_DIR, class_name)
            os.makedirs(dst_class, exist_ok=True)

            for fname in os.listdir(src_class):
                src_file = os.path.join(src_class, fname)
                dst_file = os.path.join(dst_class, fname)

                # avoid overwriting duplicates
                if os.path.exists(dst_file):
                    base, ext = os.path.splitext(fname)
                    i = 1
                    while os.path.exists(os.path.join(dst_class, f"{base}__dup{i}{ext}")):
                        i += 1
                    dst_file = os.path.join(dst_class, f"{base}__dup{i}{ext}")

                shutil.move(src_file, dst_file)

# ----------------------------
# 5) SANITY CHECK
# ----------------------------
classes = [d for d in os.listdir(MERGED_DIR) if os.path.isdir(os.path.join(MERGED_DIR, d))]
print("Merged classes:", len(classes))

def count_images(root):
    total = 0
    for c in os.listdir(root):
        p = os.path.join(root, c)
        if os.path.isdir(p):
            total += len(os.listdir(p))
    return total

print("Total images in merged:", count_images(MERGED_DIR))
print("Example class:", classes[0] if classes else None,
      "files:", (os.listdir(os.path.join(MERGED_DIR, classes[0]))[:10] if classes else None))

if len(classes) == 0:
    raise RuntimeError("MERGED_DIR has no class folders. Check your unzip results and folder structure.")


ZIP: /content/training1.zip exists: True
ZIP: /content/training2.zip exists: True
Need unzip: True
Extracting: /content/training1.zip
Extracting: /content/training2.zip
Extract root contents: ['.config', 'training1.zip', 'training2', 'training2.zip', 'resnet34_imagenet_1000_no_top.h5', 'training1', 'sample_data']
Merging into: /content/all
Merged classes: 43
Total images in merged: 39209
Example class: 00035 files: ['00033_00011.jpg', '00039_00018.jpg', '00021_00013.jpg', '00014_00019.jpg', '00027_00003.jpg', '00019_00002.jpg', '00038_00002.jpg', '00033_00028.jpg', '00024_00026.jpg', '00003_00009.jpg']


In [5]:
!pip install image-classifiers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.9 MB/s eta 0:00:00


In [6]:
from classification_models.tfkeras import Classifiers
from tensorflow.keras import layers, models

# 1. 설정 및 데이터 파이프라인
data_dir = '/content/training/'
weight_path = '/content/resnet34_custom_weights.h5' # 업로드한 가중치 파일 경로로 수정 필수
batch_size = 128
img_size = (224, 224)

ResNet34, preprocess_input = Classifiers.get('resnet34')

def preprocess_wrapper(img, label):
    return preprocess_input(img), label

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=123,
    image_size=img_size, batch_size=batch_size
).map(preprocess_wrapper, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=123,
    image_size=img_size, batch_size=batch_size
).map(preprocess_wrapper, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# 2. 빈 백본 모델 생성 (weights=None)
base_model = ResNet34(input_shape=(224, 224, 3), weights=None, include_top=False)

# 3. 직접 업로드한 가중치 주입
# 주의: 업로드한 가중치가 전체 모델용이 아니라 '백본 전용'일 경우 여기서 로드합니다.
# 레이어 이름이 미세하게 다를 경우 by_name=True 옵션이 필요할 수 있습니다.
base_model.load_weights(weight_path)

# 4. 백본 전체 동결
base_model.trainable = False

# 5. 모델 조립 (커스텀 헤드 부착)
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)

x = layers.Conv2D(64, (1, 1), padding='same', use_bias=False, name='bottleneck_conv')(x)
x = layers.BatchNormalization(name='bottleneck_bn')(x)
x = layers.Activation('relu', name='bottleneck_relu')(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2, name='head_dropout')(x)
outputs = layers.Dense(43, activation='softmax', name='predictions')(x)

model = models.Model(inputs, outputs)

# 6. 컴파일
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# history = model.fit(train_ds, validation_data=val_ds, epochs=30)

Found 39209 files belonging to 43 classes.
Using 31368 files for training.
Found 39209 files belonging to 43 classes.
Using 7841 files for validation.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional (Functional)         │ (None, 7, 7, 512)      │    21,302,473 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck_conv (Conv2D)        │ (None, 7, 7, 64)       │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck_bn                   │ (None, 7, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck_relu (Activation)    │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout (Dropout)          │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 43)             │         2,795 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,338,292 (81.40 MB)

 Trainable params: 35,691 (139.42 KB)

 Non-trainable params: 21,302,601 (81.26 MB)

In [8]:
import os
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# 1. 콜백(Callbacks) 설정
# 코랩 환경에서는 런타임이 끊길 수 있으므로 가장 좋은 가중치를 수시로 저장해야 합니다.
save_dir = '/content/saved_models'
os.makedirs(save_dir, exist_ok=True)

# 검증 정확도(val_accuracy)가 가장 높을 때의 가중치만 덮어쓰기 저장
checkpoint = ModelCheckpoint(
    filepath=os.path.join(save_dir, 'best_resnet34_head.weights.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=True, # 모델 구조는 놔두고 가중치만 가볍게 저장
    verbose=1
)

# 검증 손실(val_loss)이 5에폭 동안 개선되지 않으면 조기 종료
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True, # 종료 시 가장 성능이 좋았던 에폭의 가중치로 자동 복구
    verbose=1
)

# 검증 손실이 정체되면 학습률(Learning Rate)을 절반(0.5)으로 줄여 미세 조정 유도
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# 2. 모델 학습 실행
epochs = 30

print("🚀 백본(Stage 1~4) 동결 상태로 커스텀 헤드 학습을 시작합니다...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[checkpoint, early_stopping, reduce_lr]
)

# 3. 학습 결과 시각화 (Loss 및 Accuracy 추이 확인)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

# 정확도 그래프
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.grid(True)

# 손실 그래프
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.grid(True)

plt.show()

🚀 백본(Stage 1~4) 동결 상태로 커스텀 헤드 학습을 시작합니다...
Epoch 1/30
246/246 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.3734 - loss: 2.5540
Epoch 1: val_accuracy improved from None to 0.72593, saving model to /content/saved_models/best_resnet34_head.weights.h5

Epoch 1: finished saving model to /content/saved_models/best_resnet34_head.weights.h5
246/246 ━━━━━━━━━━━━━━━━━━━━ 88s 291ms/step - accuracy: 0.5199 - loss: 1.9354 - val_accuracy: 0.7259 - val_loss: 1.1048 - learning_rate: 0.0010
Epoch 2/30
245/246 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step - accuracy: 0.7183 - loss: 1.0661
Epoch 2: val_accuracy improved from 0.72593 to 0.83727, saving model to /content/saved_models/best_resnet34_head.weights.h5

Epoch 2: finished saving model to /content/saved_models/best_resnet34_head.weights.h5
246/246 ━━━━━━━━━━━━━━━━━━━━ 60s 244ms/step - accuracy: 0.7477 - loss: 0.9470 - val_accuracy: 0.8373 - val_loss: 0.6566 - learning_rate: 0.0010
Epoch 3/30
 98/246 ━━━━━━━━━━━━━━━━━━━━ 28s 191ms/step - accuracy: 0.7971 

KeyboardInterrupt: 

In [ ]:
import os, hashlib, random

root = "/content/data/all"
seen = set()
dups = 0
checked = 0

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    if not os.path.isdir(cls_path):
        continue
    files = os.listdir(cls_path)
    # sample up to 200 per class to keep it fast
    for f in random.sample(files, min(200, len(files))):
        p = os.path.join(cls_path, f)
        with open(p, "rb") as fp:
            h = hashlib.md5(fp.read()).hexdigest()
        checked += 1
        if h in seen:
            dups += 1
        else:
            seen.add(h)

print("Checked:", checked, " Sample duplicates:", dups)

In [ ]:
model.evaluate(val_ds, verbose=0)

In [ ]:
import numpy as np
import tensorflow as tf

y_true = []
y_pred = []

for x, y in val_ds:
    p = model.predict(x, verbose=0)
    y_true.append(y.numpy())
    y_pred.append(np.argmax(p, axis=1))

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES).numpy()
acc = (y_true == y_pred).mean()
print("Val accuracy (recomputed):", acc)
print("Confusion matrix shape:", cm.shape)

In [ ]:
import numpy as np

wrong = (y_true != y_pred)
print("Num wrong:", int(wrong.sum()), "out of", len(y_true))

# per-class accuracy
per_class = []
for c in range(NUM_CLASSES):
    mask = (y_true == c)
    if mask.sum() == 0:
        per_class.append((c, None, 0))
    else:
        per_class.append((c, float((y_pred[mask] == c).mean()), int(mask.sum())))

# show worst 10 classes (lowest accuracy), ignoring empty classes
per_class_nonempty = [(c, acc, n) for (c, acc, n) in per_class if acc is not None]
worst10 = sorted(per_class_nonempty, key=lambda x: x[1])[:10]
print("Worst 10 classes (class, acc, n):")
for row in worst10:
    print(row)